# Mini-Project 4 — Gym Workout Sessions (Bronze → Silver) — Guided v2

Clean a dirty workout-log file and save it as a Delta table.

**File:** `raw_data_4/gym_workout_sessions.csv` (397 rows x 9 cols)
**Target table:** `dev.mini_projects.gym_silver`

### How this notebook works
Every task has three parts:
1. **What to do** — the goal.
2. **Why now** — why this step comes here and what it sets up for the next step. Read this — it teaches the pipeline logic.
3. **Rough steps** — a short outline of the sub-steps. I do **not** tell you which function or method to use — you figure that out, run it, and check it.

Write your code in the empty code cell, then answer the interview questions in English in the answer cell.

### The big picture (bronze → silver)
You start from raw, dirty data (**bronze**) and end with clean, correctly-typed, deduplicated data ready for analysts (**silver**). Order matters: each step depends on the one before it. The golden rule you will feel in this project: **clean the text first, change the type second.**

---


## Task 0 — Read & explore (your bronze layer)

**What to do:** Load the CSV into a DataFrame called `gym_raw_df`. For now, read **every column as text**. Then inspect it: look at the schema, show a few rows, and get the total row count.

**Why now:** This is your raw "bronze" copy — the untouched truth. You read everything as text on purpose, so Spark does **not** guess types and silently corrupt dirty values. You can't clean what you haven't looked at, so explore before you touch anything.

**Rough steps:**
- Read the file, forcing all columns to stay as text.
- Show a sample of rows and print the schema.
- Count the rows.
- Eyeball the data: which columns look dirty? (placeholders, mixed date formats, currency symbols...)


In [ ]:
gym_raw_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/gym_workout_sessions.csv")
)
gym_raw_df.display()

In [ ]:
gym_raw_df.printSchema()

In [ ]:
len(gym_raw_df.columns)

In [ ]:
gym_raw_df.count()

### Interview questions
1. Should you let Spark guess the column types, or set them yourself? What's the trade-off?
2. How could automatic type-guessing mislead you on a messy file like this one?


1. In production I prefer defining the schema myself (schema on read). It gives me control, guarantees the types, and Spark reads the file only once instead of twice. Inference is faster to write and fine for quick exploration, but it reads the data twice and one bad value in a column — like 'ERROR' in a numeric field — makes Spark pick the wrong type, which breaks my analysis later.
2. Yes. When Spark infers types, even one bad value like 'error' or 'unknown' (not a real null) forces the whole column to become a string. So a column that should be numeric ends up as text, and then I can't run sum or avg on it without casting. Inference also only samples part of the file, so a column can look like an integer at the top but overflow or break further down


## Task 1 — Remove duplicates

**What to do:** First remove exact duplicate rows. Then handle the case where the same `session_id` appears more than once, so each session is represented only once. Check the row count before and after.

**Why now:** Do this early, before any counting or math. If the same row is counted twice, every later average, sum, or count is wrong. Get to one clean row per session first, then build on it.

**Rough steps:**
- Count rows now.
- Remove rows that are completely identical.
- Handle rows that repeat the same `session_id`.
- Count again and note how many you dropped.


In [ ]:
gym_raw_df.count()

In [ ]:
print(gym_raw_df.distinct().count())
print(gym_raw_df.count())

In [ ]:
print(gym_raw_df.select("session_id").distinct().count())
print(gym_raw_df.select("session_id").count())

In [ ]:
gym_dedup_df = gym_raw_df.dropDuplicates()

In [ ]:
print(gym_dedup_df.select("session_id").count())
print(gym_dedup_df.select("session_id").distinct().count())

In [ ]:
gym_dedup_df.display()

In [ ]:
gym_dedup_df.groupBy("session_id").count().where("count>1").display()

In [ ]:
from pyspark.sql.functions import col
gym_dedup_df.filter(col("session_id").isin("1193","1069")).display()

In [ ]:
gym_dedup_df = gym_dedup_df.dropDuplicates(["session_id"])
gym_dedup_df.select("session_id").count()

In [ ]:
gym_dedup_df.display()

### Interview questions
1. There are two different meanings of "duplicate" in this data — what are they?
2. When you drop duplicates based on a key, is it predictable which row survives? Why does that matter?
3. Is removing duplicates a lazy step, or does it trigger work right away?


1. There are two kinds of duplicates here. First, fully identical rows — every column is the same. Second, rows that share the same session_id but differ in some columns. For this training I just kept one row per session_id with dropDuplicates. Normally I'd check which values are correct and keep the reasonable ones.


2. No, it's not predictable. When you use dropDuplicates on a key, Spark keeps one row at random and drops the rest — it's non-deterministic. This matters because the duplicate rows can have different values (like duration 70 vs 94), so Spark might keep the wrong one, which hurts data quality and later analysis.


3. Removing duplicates is a lazy step (a transformation). It doesn't run until an action triggers it — like display, count, or save.

## Task 2 — Normalize the text columns

**What to do:** Make `member_name`, `city`, and `workout_type` consistent. The same value currently appears in different forms (extra spaces, different capitalization). Clean them so identical values are treated as identical.

**Why now:** You normalize text **before** you group, filter, or join on it. If "Yoga" and "yoga " stay different, a later `group by` will split one category into two and your numbers will be wrong.

**Rough steps:**
- Remove leading/trailing spaces.
- Put the text into one consistent case.
- Re-check the distinct values to confirm they collapsed.


In [ ]:
gym_dedup_df.select("member_name","city","workout_type").display()

In [ ]:
from pyspark.sql.functions import col, trim, initcap
gym_dedup_df = gym_dedup_df.withColumns({
    "member_name": trim(initcap(col("member_name"))),
    "city":trim(initcap(col("city"))),
    "workout_type": trim(initcap(col("workout_type")))
})
gym_dedup_df.limit(2).display()

In [ ]:
gym_dedup_df.printSchema()

### Interview questions
1. Give a concrete example from this data where inconsistent text would cause a wrong result later.
2. Where in a pipeline is the best place to normalize text, and why?


1. In workout_type the same value appears as "Yoga", "yoga", and "YOGA". If I don't normalize, a groupBy counts them as three separate groups, so my "sessions per workout type" numbers are wrong. The same happens with city ("Antalya" vs "ANTALYA").


2. I normalize text in the bronze-to-silver step. Bronze is the raw data; silver is the cleaned one. I do it before any groupBy, filter, or join, because those operations treat inconsistent text as different values and give wrong results.

## Task 3 — Turn fake nulls into real nulls

**What to do:** Several columns contain text placeholders like `N/A`, `UNKNOWN`, `ERROR`, or empty strings. Convert these into real nulls.

**Why now:** A placeholder pretends to be real data. It hides the fact that a value is missing and it pollutes counts (e.g. it shows up as an extra category). Convert placeholders to honest nulls **before** any numeric work or analysis, so "missing" actually looks missing.

**Rough steps:**
- Find which columns still carry placeholder tokens.
- Replace those tokens with null.
- Prove that none are left (think: which check would show zero remaining?).


In [ ]:
from pyspark.sql.functions import isnull, col
print(gym_dedup_df.filter(col("session_id").isNull()).count())
print(gym_dedup_df.filter(col("member_name").isNull()).count())
print(gym_dedup_df.filter(col("city").isNull()).count())
print(gym_dedup_df.filter(col("workout_type").isNull()).count())
print(gym_dedup_df.filter(col("duration_min").isNull()).count())
print(gym_dedup_df.filter(col("calories_burned").isNull()).count())
print(gym_dedup_df.filter(col("heart_rate_avg").isNull()).count())
print(gym_dedup_df.filter(col("session_date").isNull()).count())
print(gym_dedup_df.filter(col("membership_fee").isNull()).count())

In [ ]:
# gym_dedup_df.groupBy("duration_min").count().display() # N/A , ERROR , UNKNOWN
# gym_dedup_df.groupBy("calories_burned").count().display() # N/A , ERROR, UNKNOWN
# gym_dedup_df.groupBy("heart_rate_avg").count().display() # N/A , ERROR, UNKNOWN



In [ ]:
from pyspark.sql.functions import col, when
gym_nulled_df = gym_dedup_df.withColumns({
    "duration_min": when(col("duration_min").isin("ERROR","N/A","UNKNOWN"), None).otherwise(col("duration_min")),
    "calories_burned": when(col("calories_burned").isin("ERROR","N/A","UNKNOWN"), None).otherwise(col("calories_burned")),
    "heart_rate_avg": when(col("heart_rate_avg").isin("ERROR","N/A","UNKNOWN"), None).otherwise(col("heart_rate_avg"))
})
gym_nulled_df.display()


In [ ]:
# controlling
from pyspark.sql.functions import col
gym_nulled_df.filter(col("duration_min").isin("ERROR")).display()

### Interview questions
1. Why is a placeholder like "UNKNOWN" more dangerous than a truly empty value?
2. After converting them, how would you prove that no fake placeholders are left?


1. A placeholder looks like real data, so Spark counts it and groupBy makes it its own group. This makes the results wrong. A real null is skipped by functions like avg and count, but a placeholder hides the missing value and gets counted.

2. I run isin("ERROR","N/A","UNKNOWN").count() on each column. If it returns 0, none are left. I also check that isNull().count() went up, which proves they became real nulls.


## Task 4 — Clean the messy text BEFORE converting it (the step you were missing)

**What to do:** Some columns are still **text that only looks like a number**. The clearest example is `membership_fee`, which has values like `₺149,90` — a currency symbol plus a **comma** as the decimal separator. Clean the text so it becomes a plain number string (for example `149.90`). **Do not change the type yet** — only clean the text. Do the same kind of cleaning for any other column that has stray symbols.

**Why now (read this carefully):** This is exactly where you got stuck last time. You **cannot** turn `₺149,90` straight into a number — Spark sees the `₺` and the `,` and refuses, throwing a "cannot cast" error. Casting is the **next** task. This task only prepares the text so that the cast can succeed.
> Analogy: you wash and peel the vegetable (clean the text) **before** you cut it (cast the type). Skipping the wash is why the knife slipped last time.

**Rough steps:**
- Look at the raw values of `membership_fee` (and any other dirty text columns).
- Remove everything that isn't a digit or the decimal mark (drop the currency symbol, spaces, etc.).
- Make the decimal separator a dot, because the numeric types expect a dot, not a comma.
- Keep the result as **text** for now — you'll cast it in the next task.


In [ ]:
gym_nulled_df.select("membership_fee").display()

In [ ]:
from pyspark.sql.functions import expr, trim

gym_clean_df = gym_nulled_df.withColumns({
    "membership_fee": trim(expr("replace(replace(replace(membership_fee, '₺',''),'TL',''),',','.')"))
})
gym_clean_df.display()




### Interview questions
1. Why must you clean the text **before** changing its type, instead of casting first and cleaning later?
2. If you skip this step and cast directly, the value can become an **error** or a **silent null** depending on settings. Which outcome is worse in a daily pipeline, and why?


1. Cast needs clean text to work. If the value still has symbols like ₺ or a comma, Spark can't read it as a number. So I clean the text first, then cast. Wash the vegetable before you cut it.
2. A silent null is worse. An error stops the pipeline, so you see the problem right away and fix it. A silent null hides the problem: the row stays but the value is gone, and your numbers are wrong without any warning. In production I prefer it to fail loudly than to lose data quietly.

## Task 5 — Convert the data types

**What to do:** Now that the text is clean, give each column its correct type:
- the whole-number columns (duration, calories, heart rate),
- `membership_fee` → an exact decimal number (you cleaned it in Task 4),
- `session_date` → a real date. Careful: this column mixes **several different date formats**.

**Why now:** Correct types unlock everything downstream — math on the numbers, sorting and filtering on real dates. You do this **after** cleaning, because a cast only works on clean text (you just proved that with the fee column).

**Rough steps:**
- Choose a sensible numeric type for the counts, and an exact type for money.
- Cast the cleaned fee column.
- For the date: a single fixed format won't catch every row. Worse, a strict parser **throws an error** on the first value that doesn't match instead of returning null — so you need an approach that **tolerates a mismatch** and **tries each format in turn**, keeping the first one that works.
- Re-check: did every date convert, or are some still null (a format you missed)?


In [ ]:
from pyspark.sql.functions import  to_date, coalesce, expr

gym_clean_df = gym_clean_df.withColumns({
    "duration_min": col("duration_min").cast("int"),
    "calories_burned": col("calories_burned").cast("int"),
    "heart_rate_avg": col("heart_rate_avg").cast("int"),
    "session_date": expr("""coalesce(
        try_to_date(session_date,'yyyy-MM-dd'),
        try_to_date(session_date,'yyyy/MM/dd'),
        try_to_date(session_date,'dd-MM-yyyy'),
        try_to_date(session_date,'dd/MM/yyyy')
                                    )
    """),
    "membership_fee": col("membership_fee").cast("decimal(10,2)") 
})

gym_clean_df.display()

In [ ]:
gym_clean_df.filter(col("session_date").isNull()).count()

### Interview questions
1. For money, would you use an approximate or an exact decimal type? Why does it matter?
2. When a value can't be converted to the target type, what should happen to it — error or silent null — and how do you avoid silently losing data?
3. How do you deal with a single column that contains several different date formats?


1. Exact (decimal). Money needs to be precise. An approximate type like double creates rounding errors, for example 199.95 becomes 199.95000000000002. With decimal every cent is correct.
2. A silent null is dangerous because data disappears without any warning. To avoid losing it, I clean the text before casting, and I check the null count before and after. If new nulls appear, I know some values failed to convert and I fix the cause instead of ignoring it.
3. I use coalesce with one try_to_date per format. Each one tries a format; if it doesn't match it returns null, so coalesce keeps the first one that works. I use try_to_date (not to_date) because the strict version throws an error, while try_to_date returns null and lets coalesce move on.


In [ ]:
gym_clean_df.printSchema()

## Task 6 — Compute new columns

**What to do:** Add at least two new columns:
- `calories_per_min` — calories burned divided by duration (rounded sensibly).
- `intensity` — a label such as low / medium / high, based on a rule you decide.

Try writing this in more than one way if you can.

**Why now:** Derived metrics come **after** the types are correct. If you computed `calories_per_min` while the columns were still text, the math would fail or give garbage. Clean + typed data first, then derive.


In [ ]:
from pyspark.sql.functions import expr,col,try_divide, round
gym_new_clean_df = gym_clean_df.withColumns({
    "calories_per_min": expr("round(try_divide(calories_burned, duration_min))"),
    "intensity": when(col("calories_per_min") >10, "high")
                 .when((col("calories_per_min")> 7) & (col("calories_per_min")< 10), "normal")
                 .otherwise("low")
})

gym_new_clean_df.display()

### Interview questions
1. Is adding a computed column a lazy or an immediate operation?
2. If a new column depends on another new column you just created in the same step, what do you have to watch out for?


1. It is lazy. withColumn is a transformation, so Spark only builds the plan and waits. Nothing runs until an action like display, count, or write is called.
2. In a single withColumns, all columns are built from the input DataFrame, so a new column can't see another new column made in the same step. The fix is to create the first column in one step, then use it in a second step — or repeat the raw formula instead of pointing to the new column.



## Task 7 — Filter out invalid rows (business rules)

**What to do:** Keep only the rows that make sense: `duration_min` greater than 0, `calories_burned` greater than 0, and the key fields not null. Report how many rows you had before and after, and what percentage you dropped.

**Why now:** Business rules go near the **end**, once the data is clean and typed. Filtering on trustworthy values means you drop genuinely bad rows, not rows that only looked bad because of dirty text.

**Rough steps:**
- Express the rule as "keep the good rows" (not "delete the bad rows").
- Combine the conditions carefully.
- Count before and after; compute the percentage dropped.


In [ ]:
gym_final_df = gym_new_clean_df.filter((col("duration_min")>0) & (col("calories_burned")>0))
gym_final_df.display()

### Interview questions
1. Why is it safer to express the rule as "keep the good rows" instead of "delete the bad rows"?
2. When you combine several conditions together, what's a common mistake that silently changes the result?
3. How much data did you drop — and how would you decide whether that's acceptable?


1. Because of nulls. A condition like duration_min > 0 is clear, but duration_min <= 0 can miss null rows (null is not true or false). When you keep the good rows, anything you are not sure about is dropped, so bad data can't slip through.
2. Forgetting the parentheses. & and | bind tighter than > or ==, so without parentheses around each condition the logic is wrong and Spark may even fail or give the wrong rows. Always wrap each condition: (a > 0) & (b > 0).
3. I count before and after and look at the percentage. A small drop (like 1-2%) of clearly broken rows is fine. If a big share disappears, I stop and check why — maybe my cleaning has a bug and I'm throwing away good data, not bad data.



## Task 8 — Save the silver table

**What to do:** Write your cleaned DataFrame as the Delta table `dev.mini_projects.gym_silver`. Make it so re-running the whole notebook produces the same result (no duplicated rows on a second run).

**Why now:** This is your **silver** layer — clean, typed, deduplicated, ready for analysts to query. It's the payoff of every step above. Making it idempotent means a daily rerun won't double the data.

**Rough steps:**
- Write the DataFrame as a managed Delta table with that name.
- Make the write idempotent (a rerun replaces, it doesn't append on top).


In [ ]:
gym_final_df.write.mode("overwrite").saveAsTable("dev.mini_projects.gym_silver")

In [ ]:
spark.table("dev.mini_projects.gym_silver").display()

### Interview questions
1. What does "idempotent" mean, and why does it matter for a job that reruns every day?
2. What's the difference between overwriting and appending when you save?
3. What's the difference between saving a table and creating a temporary view?


1. Idempotent means running the job once or many times gives the same result. It matters because a daily job reruns every day. With overwrite the table is replaced each time, so the data stays correct. Without it (using append), every rerun would add the rows again and double the data.
2. Overwrite deletes the old data and writes the new data. Append keeps the old data and adds the new rows on top. For a full clean reload I use overwrite; for adding new daily data I use append.
3. A saved table is permanent and stored on disk (as Delta), so other people and other notebooks can query it later. A temporary view only lives in the current Spark session and disappears when the session ends. A table is real storage; a temp view is just a name for a query.